# Init-scale sweep — training dynamics & representations

Compares how `init_scale` (global weight multiplier at initialisation) affects:
- **Training dynamics**: how population activity evolves over training
- **Representations**: trial-aligned activation heatmaps per stim condition
- **Weight structure**: input / recurrent / readout weight matrices vs stim activation
- **Performance & selectivity**: summary metrics across scales

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pickle, glob, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgs
import pandas as pd
import torch

sys.path.insert(0, str(Path("..").resolve()))
from cxval.models import RNN, ActorCritic
from cxval.envs import TaskEnv
from cxval.agents import Agent
from cxval.analysis import compute_unit_tuning, preferred_value_proportions

SWEEP_DIR       = Path("../results/01_06_26_sweep_init_scale")
INIT_SCALE_VALS = [0.01, 0.05, 0.1, 0.5, 1.0]
DEVICE          = torch.device("cpu")
SEED_PICK       = 42      # representative seed shown in per-run plots
ITI_TAIL        = 3       # timesteps before stim onset to include
SI_THRESHOLD    = 0.1
SILENT_THR      = 1e-4
STIM_COLORS     = ["#5577aa", "#cc7733", "#44aa55"]   # low / mid / high

out_dir = SWEEP_DIR / "figures"
out_dir.mkdir(exist_ok=True)
print(f"SWEEP_DIR: {SWEEP_DIR}")
print(f"out_dir  : {out_dir}")

In [ ]:
# ── Load all vis_data.pkl files, index by (init_scale, seed) ──────────────
_all_pkls = sorted(SWEEP_DIR.glob("*/vis_data.pkl"))
print(f"Found {len(_all_pkls)} runs")

runs = {}   # (init_scale, seed) -> vd
for _p in _all_pkls:
    with open(_p, "rb") as _f:
        _vd = pickle.load(_f)
    _key = (float(_vd["init_scale"]), int(_vd["seed"]))
    runs[_key] = _vd

# Summary DataFrame
rows = []
for (_is, _seed), _vd in runs.items():
    _psa = _vd.get("psa_results", {}).get(0, {})
    rows.append(dict(
        init_scale=_is, seed=_seed,
        spearman_r=_vd.get("spearman_r", np.nan),
        psa_score=_psa.get("psa_score", np.nan),
        lick_high=_psa.get("high_lick", np.nan),
        lick_mid=_psa.get("mid_lick", np.nan),
        lick_low=_psa.get("low_lick", np.nan),
        frac_selective=np.nan,
        mean_init_act=_vd.get("mean_init_act", np.nan),
        mean_trained_act=_vd.get("mean_trained_act", np.nan),
        run_dir=str(_p.parent),
    ))
df = pd.DataFrame(rows).sort_values(["init_scale", "seed"])
print(df[["init_scale","seed","spearman_r","psa_score","lick_high","lick_mid","lick_low",
          "mean_init_act","mean_trained_act"]].to_string(index=False))

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────

def _t(p):
    return np.array(p.detach().cpu().tolist(), dtype=np.float32)

def load_model(vd, state_dict, device=DEVICE):
    obs_dim       = vd["infer_states"].shape[1] + 2
    readout_frac  = vd.get("readout_fraction", 1.0)
    bb  = RNN(input_size=obs_dim, hidden_size=vd["hidden_size"], output_size=1)
    ac  = ActorCritic(backbone=bb, num_actions=2, readout_fraction=readout_frac)
    ac.load_state_dict(state_dict)
    ac.eval()
    return ac.to(device)

def run_inference(ac, vd, device=DEVICE):
    """Return hidden_np (T, H) for the inference state sequence in vd."""
    env_kw = {k: vd[k] for k in
              ("reward_lick","reward_no_lick","reward_lick_miss","lick_cost") if k in vd}
    env = TaskEnv(states=vd["infer_states"],
                  reward_availability=vd["infer_reward_availability"], **env_kw)
    agent = Agent(ac, device=device)
    agent.reset()
    obs, _ = env.reset()
    hs = []
    if any(torch.isnan(p).any() for p in ac.parameters()):
        return None   # diverged checkpoint
    done = False
    try:
        while not done:
            action, _, _ = agent.act(obs)
            hs.append(np.array(agent.hidden.detach().squeeze(0).cpu().tolist(), dtype=np.float32))
            obs, _, done, _, _ = env.step(action)
    except (ValueError, RuntimeError):
        return None   # NaN/inf in logits — diverged
    return np.array(hs)

def run_inference_full(ac, vd, device=DEVICE):
    """Return (hidden_np, lick_probs) where lick_probs[t] = P(lick) at timestep t.

    TaskEnv.LICK = 0, so lick probability = softmax(logits)[0].
    Returns (None, None) if the model has diverged.
    """
    env_kw = {k: vd[k] for k in
              ("reward_lick","reward_no_lick","reward_lick_miss","lick_cost") if k in vd}
    env = TaskEnv(states=vd["infer_states"],
                  reward_availability=vd["infer_reward_availability"], **env_kw)
    obs, _ = env.reset()
    hidden = None
    hs, lick_ps = [], []
    if any(torch.isnan(p).any() for p in ac.parameters()):
        return None, None
    done = False
    try:
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                logits, _, hidden = ac.step(obs_t, hidden)
            lick_ps.append(float(torch.softmax(logits, dim=-1)[0, 0]))  # P(lick=action 0)
            hs.append(np.array(hidden.detach().squeeze(0).cpu().tolist(), dtype=np.float32))
            action = torch.distributions.Categorical(logits=logits).sample().item()
            obs, _, done, _, _ = env.step(action)
    except (ValueError, RuntimeError):
        return None, None
    return np.array(hs), np.array(lick_ps, dtype=np.float32)

def build_snippets(hidden_np, struct, stim_arr, stim_idx, iti_tail):
    """Return (n_trials, win_len, H) array of trial-aligned snippets for one stim."""
    stim_len  = struct[0]["stim_window"][1] - struct[0]["stim_window"][0]
    rew_len   = struct[0]["reward_window"][1] - struct[0]["reward_window"][0]
    win_len   = iti_tail + stim_len + rew_len
    snips = []
    for ti, tr in enumerate(struct):
        if stim_arr[ti] != stim_idx:
            continue
        ss, se = tr["stim_window"]
        rs, re = tr["reward_window"]
        iti_s  = max(0, ss - iti_tail)
        pad    = iti_tail - (ss - iti_s)
        chunk  = np.concatenate([
            np.zeros((pad, hidden_np.shape[1])),
            hidden_np[iti_s:se],
            hidden_np[rs:re],
        ], axis=0)
        snips.append(chunk[:win_len])
    return np.stack(snips) if snips else None, win_len, stim_len, iti_tail

def lick_snippets(lick_probs, struct, stim_arr, stim_idx, iti_tail):
    """Like build_snippets but for a 1-D lick_probs array. Returns (n_trials, win_len)."""
    stim_len = struct[0]["stim_window"][1] - struct[0]["stim_window"][0]
    rew_len  = struct[0]["reward_window"][1] - struct[0]["reward_window"][0]
    win_len  = iti_tail + stim_len + rew_len
    snips = []
    for ti, tr in enumerate(struct):
        if stim_arr[ti] != stim_idx:
            continue
        ss, se = tr["stim_window"]
        rs, re = tr["reward_window"]
        iti_s  = max(0, ss - iti_tail)
        pad    = iti_tail - (ss - iti_s)
        chunk  = np.concatenate([
            np.full(pad, np.nan),
            lick_probs[iti_s:se],
            lick_probs[rs:re],
        ])
        snips.append(chunk[:win_len])
    return np.stack(snips) if snips else None

def stim_mean_act(hidden_np, stim_arr, struct, n_stimuli):
    """Return (H, n_stim) mean activation during stim window per stimulus."""
    acts = []
    for si in range(n_stimuli):
        chunks = np.concatenate([
            hidden_np[t["stim_window"][0]:t["stim_window"][1]]
            for t, m in zip(struct, stim_arr == si) if m
        ], axis=0)
        acts.append(chunks.mean(axis=0))
    return np.stack(acts, axis=1)


## §1 Training dynamics (checkpoint-based)

For each `init_scale`, loads each checkpoint and runs inference to track
how mean population activity in ITI / stim / reward windows evolves over training.
Uses the representative seed (`SEED_PICK=42`).

In [ ]:
# ── Training dynamics: pop activity + lick prob per period per stim ────────
# One row per init_scale; three columns (ITI / Stim / Reward).
# Mean ± SEM shaded across all available seeds.
#
# Results are cached to disk so re-running the cell is instant.
# Set FORCE_RECOMPUTE = True to re-run from scratch.

FORCE_RECOMPUTE = False
_CACHE_FILE     = SWEEP_DIR / "_ckpt_dynamics_cache.pkl"

_period_names = ["ITI", "Stim", "Reward"]
_all_seeds    = sorted({s for (_, s) in runs.keys()})

if not FORCE_RECOMPUTE and _CACHE_FILE.exists():
    print(f"Loading cached dynamics from {_CACHE_FILE}")
    with open(_CACHE_FILE, "rb") as _f:
        _cache = pickle.load(_f)
    _all_dyn   = _cache["all_dyn"]
    _ckpt_grid = _cache["ckpt_grid"]
    print("Done.")
else:
    _all_dyn   = {}
    _ckpt_grid = {}

    for _isv in INIT_SCALE_VALS:
        _all_dyn[_isv] = {}
        for _seed in _all_seeds:
            _key = (_isv, _seed)
            if _key not in runs:
                continue
            _vd       = runs[_key]
            _run_dir  = SWEEP_DIR / _vd["run_id"]
            _ckpt_dir = _run_dir / "checkpoints"
            _ckpts    = sorted(_ckpt_dir.glob("checkpoint_*.pt"))
            if not _ckpts:
                print(f"  WARNING: no checkpoints for init_scale={_isv} seed={_seed}")
                continue

            _struct   = _vd["infer_trial_structure"]
            _stim_arr = np.array([t["stimulus"] for t in _struct])
            _n_stim   = _vd["n_stimuli"]
            _stim_len = _struct[0]["stim_window"][1] - _struct[0]["stim_window"][0]
            _rew_len  = _struct[0]["reward_window"][1] - _struct[0]["reward_window"][0]
            _periods_sl = [
                slice(0, ITI_TAIL),
                slice(ITI_TAIL, ITI_TAIL + _stim_len),
                slice(ITI_TAIL + _stim_len, None),
            ]
            _ckpt_trials = [int(_c.stem.split("_")[1]) for _c in _ckpts]
            if _isv not in _ckpt_grid:
                _ckpt_grid[_isv] = [0] + _ckpt_trials   # 0 = untrained

            _results      = {_si: [[] for _ in _period_names] for _si in range(_n_stim)}
            _lick_results = {_si: [] for _si in range(_n_stim)}

            # ── trial 0: untrained model ──────────────────────────────────────
            _init_path = _run_dir / "model_init.pt"
            if _init_path.exists():
                _sd_init_dyn    = torch.load(_init_path, map_location="cpu")
                _ac_init_dyn    = load_model(_vd, _sd_init_dyn)
                _hid_init_dyn, _lp_init_dyn = run_inference_full(_ac_init_dyn, _vd)
            else:
                _hid_init_dyn, _lp_init_dyn = None, None

            for _si in range(_n_stim):
                if _hid_init_dyn is not None:
                    _snips0, *_ = build_snippets(_hid_init_dyn, _struct, _stim_arr, _si, ITI_TAIL)
                    if _snips0 is not None:
                        _m0 = _snips0.mean(axis=0)
                        for _pi, _sl in enumerate(_periods_sl):
                            _results[_si][_pi].append(float(_m0[_sl].mean()))
                    else:
                        for _pi in range(len(_period_names)):
                            _results[_si][_pi].append(np.nan)
                    _ls0 = lick_snippets(_lp_init_dyn, _struct, _stim_arr, _si, ITI_TAIL)
                    _lick_results[_si].append(
                        float(np.nanmean(_ls0[:, ITI_TAIL:ITI_TAIL + _stim_len]))
                        if _ls0 is not None else np.nan
                    )
                else:
                    for _pi in range(len(_period_names)):
                        _results[_si][_pi].append(np.nan)
                    _lick_results[_si].append(np.nan)

            print(f"  init_scale={_isv} seed={_seed}: {len(_ckpts)} ckpts...", end=" ", flush=True)
            for _ckpt_f in _ckpts:
                _sd       = torch.load(_ckpt_f, map_location="cpu")
                _ac       = load_model(_vd, _sd)
                _hid, _lp = run_inference_full(_ac, _vd)

                if _hid is None:   # diverged — fill with NaN
                    for _si in range(_n_stim):
                        for _pi in range(len(_period_names)):
                            _results[_si][_pi].append(np.nan)
                        _lick_results[_si].append(np.nan)
                    continue

                for _si in range(_n_stim):
                    _snips, *_ = build_snippets(_hid, _struct, _stim_arr, _si, ITI_TAIL)
                    if _snips is None:
                        for _pi in range(len(_period_names)):
                            _results[_si][_pi].append(np.nan)
                        _lick_results[_si].append(np.nan)
                        continue
                    _mean = _snips.mean(axis=0)   # (win_len, H)
                    for _pi, _sl in enumerate(_periods_sl):
                        _results[_si][_pi].append(float(_mean[_sl].mean()))

                    # mean P(lick) during stim window, averaged across trials
                    _lsnips = lick_snippets(_lp, _struct, _stim_arr, _si, ITI_TAIL)
                    if _lsnips is not None:
                        _lick_results[_si].append(float(
                            np.nanmean(_lsnips[:, ITI_TAIL:ITI_TAIL + _stim_len])
                        ))
                    else:
                        _lick_results[_si].append(np.nan)

            print("done")
            _all_dyn[_isv][_seed] = {
                "results":      _results,
                "lick_results": _lick_results,
                "n_stim":       _n_stim,
                "stimuli":      _vd["stimuli"],
            }

    with open(_CACHE_FILE, "wb") as _f:
        pickle.dump({"all_dyn": _all_dyn, "ckpt_grid": _ckpt_grid}, _f)
    print(f"\nCached to {_CACHE_FILE}")


# ── Helpers ───────────────────────────────────────────────────────────────
def _mean_sem(vals_per_seed):
    arr = np.array(vals_per_seed, dtype=float)
    m   = np.nanmean(arr, axis=0)
    s   = np.nanstd(arr,  axis=0) / np.sqrt(np.sum(np.isfinite(arr), axis=0).clip(1))
    return m, s

# Y-limits: 95th percentile per (scale, period) to exclude divergence spikes
_period_ymaxes = {}
for _isv in INIT_SCALE_VALS:
    _ymaxes_isv = []
    for _pi in range(len(_period_names)):
        _vals = [v for _dyn in _all_dyn.get(_isv, {}).values()
                 for _si in range(_dyn["n_stim"])
                 for v in _dyn["results"][_si][_pi] if np.isfinite(v)]
        _ymaxes_isv.append(np.nanpercentile(_vals, 95) * 1.3 if _vals else 1.0)
    _period_ymaxes[_isv] = _ymaxes_isv


# ── Plot: population activity ─────────────────────────────────────────────
_n_scales = len(INIT_SCALE_VALS)
fig, axes = plt.subplots(_n_scales, 3, figsize=(13, 3.0 * _n_scales),
                         sharey=False, sharex=False)
if _n_scales == 1:
    axes = axes.reshape(1, 3)

for _ri, _isv in enumerate(INIT_SCALE_VALS):
    _seed_dyn    = _all_dyn.get(_isv, {})
    _ckpt_trials = _ckpt_grid.get(_isv, [])
    if not _seed_dyn or not _ckpt_trials:
        for ax in axes[_ri]:
            ax.text(0.5, 0.5, f"no data for scale={_isv}", ha="center", va="center",
                    transform=ax.transAxes)
        continue

    _any_dyn       = next(iter(_seed_dyn.values()))
    _n_stim        = _any_dyn["n_stim"]
    _stimuli       = _any_dyn["stimuli"]
    _n_seeds_avail = len(_seed_dyn)

    for _pi, (_pname, ax) in enumerate(zip(_period_names, axes[_ri])):
        for _si in range(_n_stim):
            _m, _se = _mean_sem([_seed_dyn[_s]["results"][_si][_pi] for _s in _seed_dyn])
            _col = STIM_COLORS[_si % len(STIM_COLORS)]
            ax.plot(_ckpt_trials, _m, color=_col, lw=1.8,
                    label=_stimuli[_si], marker="o", ms=3)
            ax.fill_between(_ckpt_trials, _m - _se, _m + _se,
                            color=_col, alpha=0.20, linewidth=0)
        ax.set_ylim(0, _period_ymaxes[_isv][_pi])
        ax.set_xlabel("Training trial", fontsize=8)
        ax.set_title(f"{_pname} period", fontsize=9) if _ri == 0 else None
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if _pi == 0:
            ax.set_ylabel(f"init_scale={_isv}\nMean pop act", fontsize=8)
        if _ri == _n_scales - 1 and _pi == 0:
            ax.legend(fontsize=7, loc="upper right",
                      title=f"n={_n_seeds_avail} seeds", title_fontsize=6)

fig.suptitle(
    f"Population activity over training  [mean ± SEM, n≤{len(_all_seeds)} seeds per scale]\n"
    f"Y-limits: 95th percentile per scale to exclude divergence spikes",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fig.savefig(out_dir / "fig1_training_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Plot: lick probability over training ─────────────────────────────────
# Same layout as the activity plot above; one row per init_scale.
# Each line is mean P(lick) during the stim window for one stimulus condition,
# averaged across trials and seeds.  Ideal policy dashed lines at 0 / 0.5 / 1.

fig, axes = plt.subplots(_n_scales, 1, figsize=(9, 2.8 * _n_scales), sharex=False)
if _n_scales == 1:
    axes = [axes]

for _ri, _isv in enumerate(INIT_SCALE_VALS):
    ax           = axes[_ri]
    _seed_dyn    = _all_dyn.get(_isv, {})
    _ckpt_trials = _ckpt_grid.get(_isv, [])
    if not _seed_dyn or not _ckpt_trials:
        ax.text(0.5, 0.5, f"no data for scale={_isv}", ha="center", va="center",
                transform=ax.transAxes)
        continue

    _any_dyn = next(iter(_seed_dyn.values()))
    _n_stim  = _any_dyn["n_stim"]
    _stimuli = _any_dyn["stimuli"]

    # ideal lick probs from value matrix (stim 0=low, 1=mid, 2=high for 3-stim task)
    _ideal = [0.0, 0.5, 1.0] if _n_stim == 3 else [np.nan] * _n_stim

    for _si in range(_n_stim):
        _m, _se = _mean_sem([_seed_dyn[_s]["lick_results"][_si] for _s in _seed_dyn])
        _col    = STIM_COLORS[_si % len(STIM_COLORS)]
        ax.plot(_ckpt_trials, _m, color=_col, lw=1.8,
                label=_stimuli[_si], marker="o", ms=3)
        ax.fill_between(_ckpt_trials, _m - _se, _m + _se,
                        color=_col, alpha=0.20, linewidth=0)
        if np.isfinite(_ideal[_si]):
            ax.axhline(_ideal[_si], color=_col, lw=0.8, ls=":", alpha=0.5)

    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Training trial", fontsize=8)
    ax.set_ylabel(f"init_scale={_isv}\nP(lick | stim)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    if _ri == 0:
        ax.legend(fontsize=7, loc="upper right",
                  title=f"n={len(_seed_dyn)} seeds", title_fontsize=6)
        ax.set_title("Mean P(lick) during stim window  [dotted = ideal policy]", fontsize=9)

fig.suptitle(
    f"Lick probability over training  [mean ± SEM, n≤{len(_all_seeds)} seeds per scale]",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fig.savefig(out_dir / "fig1b_lick_prob_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()


## §2 Trial-aligned heatmaps — trained model

One row of subplots per `init_scale`, three columns per stimulus condition.
Neuron order is fixed by the sort index from the first (0.01) run.

In [ ]:
# ── Compute sort index from SEED_PICK of the smallest scale run ────────────
_ref_vd  = runs.get((INIT_SCALE_VALS[0], SEED_PICK))
if _ref_vd is None:
    _ref_vd = next(v for (is_, s_), v in runs.items() if s_ == SEED_PICK)

_ref_hid  = np.array(_ref_vd["infer_activations"]["hidden_states"])
_ref_act  = _ref_vd["infer_activations"]
_ref_stim = _ref_act["stimulus"]
_ref_stru = _ref_act["trial_structure"]
_n_stim   = _ref_vd["n_stimuli"]
_stimuli  = _ref_vd["stimuli"]
_H        = _ref_vd["hidden_size"]

# Tuning for sort
_ref_stim_h  = _ref_act["stim_hidden"]     # (n_trials, stim_ts, H)
_mean_ctx    = np.zeros((_H, _n_stim))
for _si in range(_n_stim):
    _m = _ref_stim == _si
    _mean_ctx[:, _si] = _ref_stim_h[_m].mean(axis=(0, 1))

_max_abs   = _mean_ctx.max(axis=1)
_si_range  = _mean_ctx.max(axis=1) - _mean_ctx.min(axis=1)
_si_sum    = np.abs(_mean_ctx.sum(axis=1))
_si_score  = np.where(_si_sum > SILENT_THR, _si_range / (_si_sum + 1e-12), 0.0)
_pref_stim = np.argmax(_mean_ctx, axis=1)
_sel_mask  = (_si_score >= SI_THRESHOLD) & (_max_abs >= SILENT_THR)

# Sort: selective groups by preferred stim (SI desc), then non-selective
_grp_idx, _grp_sz, _grp_lbl = [], [], []
for _si in range(_n_stim):
    _g = np.where(_sel_mask & (_pref_stim == _si))[0]
    _g = _g[np.argsort(-_si_score[_g])]
    _grp_idx.append(_g); _grp_sz.append(len(_g)); _grp_lbl.append(_stimuli[_si])
_ns = np.where(~_sel_mask)[0][np.argsort(-_si_score[np.where(~_sel_mask)[0]])]
_grp_idx.append(_ns); _grp_sz.append(len(_ns)); _grp_lbl.append("non-sel")
_SORT_IDX = np.concatenate(_grp_idx).astype(int)

# Boundary positions
_boundaries = []
_cursor = 0
for _gi, _sz in enumerate(_grp_sz):
    _cursor += _sz
    if _gi < len(_grp_sz) - 1:
        _boundaries.append(_cursor - 0.5)

print(f"Sort index computed from init_scale={INIT_SCALE_VALS[0]} seed={SEED_PICK}")
print(f"Groups: {list(zip(_grp_lbl, _grp_sz))}")

In [ ]:
# ── Trial-aligned heatmaps: trained model, all init_scale values ───────────
_stim_len  = _ref_stru[0]["stim_window"][1] - _ref_stru[0]["stim_window"][0]
_rew_len   = _ref_stru[0]["reward_window"][1] - _ref_stru[0]["reward_window"][0]
_win_len   = ITI_TAIL + _stim_len + _rew_len

# x-axis labels
_tick_pos = [0, ITI_TAIL, ITI_TAIL + _stim_len, _win_len - 1]
_x_labels = {0: "ITI start", ITI_TAIL: "stim on",
             ITI_TAIL + _stim_len: "rew on", _win_len - 1: "end"}

_n_scales = len(INIT_SCALE_VALS)
fig, _axes_all = plt.subplots(
    _n_scales * 2, _n_stim + 1,
    figsize=(3.5 * _n_stim + 0.5, 4.5 * _n_scales),
    gridspec_kw={"height_ratios": ([4, 1.2] * _n_scales),
                 "width_ratios": [1.0] * _n_stim + [0.04]},
    squeeze=False,
)
# Hide all colorbar-column lower panels up front
for _ri in range(_n_scales):
    _axes_all[_ri * 2 + 1, _n_stim].set_visible(False)

for _ri, _isv in enumerate(INIT_SCALE_VALS):
    _key = (_isv, SEED_PICK)
    _row_h  = _ri * 2
    _row_p  = _ri * 2 + 1
    _cbar_ax = _axes_all[_row_h, _n_stim]

    if _key not in runs:
        _axes_all[_row_h, 0].text(0.5, 0.5, f"missing: scale={_isv}", ha="center", va="center",
                                   transform=_axes_all[_row_h, 0].transAxes)
        continue

    _vd       = runs[_key]
    _hid      = np.array(_vd["infer_activations"]["hidden_states"])
    _struct   = _vd["infer_trial_structure"]
    _stim_arr = np.array([t["stimulus"] for t in _struct])

    _all_snips = []
    for _si in range(_n_stim):
        _snips, *_ = build_snippets(_hid, _struct, _stim_arr, _si, ITI_TAIL)
        _all_snips.append(_snips)

    _all_vals = np.concatenate([s.mean(axis=0) for s in _all_snips if s is not None])
    _vmax = float(np.nanpercentile(_all_vals, 99)) if np.isfinite(_all_vals).any() else 1.0
    _ymax = (max(s.mean(axis=(0, 2)).max() for s in _all_snips if s is not None) * 1.15
             if any(s is not None for s in _all_snips) else 1.0)

    _im = None
    for _si in range(_n_stim):
        _ax_h = _axes_all[_row_h, _si]
        _ax_p = _axes_all[_row_p, _si]
        _snips = _all_snips[_si]

        if _snips is None:
            _ax_h.set_visible(False); _ax_p.set_visible(False)
            continue

        _ms = _snips.mean(axis=0)[:, _SORT_IDX].T
        _im = _ax_h.imshow(_ms, aspect="auto", cmap="hot",
                           vmin=0, vmax=_vmax, interpolation="nearest")
        _ax_h.axvline(ITI_TAIL - 0.5, color="white", lw=1.2, ls="--", alpha=0.6)
        _ax_h.axvline(ITI_TAIL + _stim_len - 0.5, color="white", lw=1.2, ls="--", alpha=0.6)
        for _by in _boundaries:
            _ax_h.axhline(_by, color="white", lw=0.8, ls=":", alpha=0.7)
        _ax_h.set_xticks([])
        _ax_h.set_xlim(-0.5, _win_len - 0.5)
        _ax_h.set_title(f"{_stimuli[_si]}\n(n={len(_snips)})", fontsize=8) if _ri == 0 else None
        if _si == 0:
            _ax_h.set_ylabel(f"init_scale={_isv}\n(n={_H} units)", fontsize=8)

        _pop = _snips.mean(axis=(0, 2))
        _ax_p.plot(range(_win_len), _pop, color=STIM_COLORS[_si % len(STIM_COLORS)], lw=1.5)
        _ax_p.axvline(ITI_TAIL - 0.5, color="gray", lw=1, ls="--", alpha=0.6)
        _ax_p.axvline(ITI_TAIL + _stim_len - 0.5, color="gray", lw=1, ls="--", alpha=0.6)
        _ax_p.set_ylim(0, _ymax)
        _ax_p.tick_params(labelsize=7)
        _ax_h.spines[["top","right"]].set_visible(False)
        _ax_p.spines[["top","right"]].set_visible(False)
        if _ri == _n_scales - 1:
            _ax_p.set_xticks(_tick_pos)
            _ax_p.set_xticklabels([_x_labels[p] for p in _tick_pos], fontsize=7, rotation=35)
        else:
            _ax_p.set_xticks([])
        if _si == 0:
            _ax_p.set_ylabel("Pop avg act", fontsize=7)

    if _im is not None:
        plt.colorbar(_im, cax=_cbar_ax, label="activation" if _ri == 0 else "")
        _cbar_ax.tick_params(labelsize=6)

fig.suptitle(
    f"Trial-aligned activations — trained model  [seed={SEED_PICK}]\n"
    f"Neurons sorted by selectivity (from init_scale={INIT_SCALE_VALS[0]})",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fig.savefig(out_dir / "fig2_trained_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

## §3 Untrained vs trained comparison per init_scale

For each scale: pop-average activity (3 stimuli) at init and after training, side by side.

In [ ]:
# ── Untrained vs trained: pop activity comparison per init_scale ───────────
# Shared y-axis across all rows so magnitudes are directly comparable.

_n_scales = len(INIT_SCALE_VALS)

# First pass: collect all pop-average traces to find global y-max
_cache_ut = {}   # _isv -> {panel_label: {si: pop array}}

for _isv in INIT_SCALE_VALS:
    _key = (_isv, SEED_PICK)
    if _key not in runs:
        continue
    _vd     = runs[_key]
    _struct = _vd["infer_trial_structure"]
    _stim_a = np.array([t["stimulus"] for t in _struct])
    _hid_tr = np.array(_vd["infer_activations"]["hidden_states"])

    _run_dir   = SWEEP_DIR / _vd["run_id"]
    _init_path = _run_dir / "model_init.pt"
    if not _init_path.exists():
        continue

    _sd_init = torch.load(_init_path, map_location="cpu")
    _ac_init = load_model(_vd, _sd_init)
    _hid_un  = run_inference(_ac_init, _vd)

    _stim_len = _struct[0]["stim_window"][1] - _struct[0]["stim_window"][0]
    _rew_len  = _struct[0]["reward_window"][1] - _struct[0]["reward_window"][0]
    _win_len  = ITI_TAIL + _stim_len + _rew_len

    _cache_ut[_isv] = {
        "hid_un": _hid_un, "hid_tr": _hid_tr,
        "struct": _struct, "stim_a": _stim_a,
        "stimuli": _vd["stimuli"], "n_stim": _vd["n_stimuli"],
        "stim_len": _stim_len, "win_len": _win_len,
    }

# Global y-max across all runs
_global_ymax = 0.0
for _d in _cache_ut.values():
    for _hid in (_d["hid_un"], _d["hid_tr"]):
        for _si in range(_d["n_stim"]):
            _snips, *_ = build_snippets(_hid, _d["struct"], _d["stim_a"], _si, ITI_TAIL)
            if _snips is not None:
                _global_ymax = max(_global_ymax, float(_snips.mean(axis=(0,2)).max()))
_global_ymax *= 1.15

fig, axes = plt.subplots(
    _n_scales, _n_stim * 2,
    figsize=(3.0 * _n_stim * 2, 3.0 * _n_scales),
    sharey=True, sharex=False,
)
if _n_scales == 1:
    axes = axes.reshape(1, -1)

for _ri, _isv in enumerate(INIT_SCALE_VALS):
    if _isv not in _cache_ut:
        axes[_ri, 0].text(0.5, 0.5, f"missing: scale={_isv}", ha="center", va="center",
                          transform=axes[_ri, 0].transAxes)
        continue
    _d = _cache_ut[_isv]

    for _panel, (_label, _hid) in enumerate([("init", _d["hid_un"]), ("trained", _d["hid_tr"])]):
        for _si in range(_d["n_stim"]):
            _col = _panel * _d["n_stim"] + _si
            _ax  = axes[_ri, _col]
            _snips, *_ = build_snippets(_hid, _d["struct"], _d["stim_a"], _si, ITI_TAIL)
            if _snips is None:
                _ax.set_visible(False)
                continue
            _pop = _snips.mean(axis=(0, 2))
            _ax.plot(range(_d["win_len"]), _pop,
                     color=STIM_COLORS[_si % len(STIM_COLORS)], lw=1.5)
            _ax.axvline(ITI_TAIL - 0.5, color="gray", lw=1, ls="--", alpha=0.5)
            _ax.axvline(ITI_TAIL + _d["stim_len"] - 0.5, color="gray", lw=1, ls="--", alpha=0.5)
            _ax.set_ylim(0, _global_ymax)
            _ax.tick_params(labelsize=7)
            _ax.set_xticks([])
            _ax.spines[["top","right"]].set_visible(False)
            if _ri == 0:
                _ax.set_title(f"{_label}\n{_d['stimuli'][_si]}", fontsize=8)
            if _col == 0:
                _ax.set_ylabel(f"init_scale={_isv}\nPop avg act", fontsize=8)

    # Divider between init and trained panels
    axes[_ri, _d["n_stim"] - 1].spines["right"].set_linewidth(2)
    axes[_ri, _d["n_stim"] - 1].spines["right"].set_color("navy")
    axes[_ri, _d["n_stim"] - 1].spines["right"].set_visible(True)

fig.suptitle(
    f"Untrained vs trained population activity  [seed={SEED_PICK}]  (shared y-axis)",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fig.savefig(out_dir / "fig3_untrained_vs_trained.png", dpi=150, bbox_inches="tight")
plt.show()


## §4 Weight matrices + stim activations per init_scale

For each `init_scale` (representative seed): stim mean activation aligned with
input weights, recurrent weights, and readout weights — both untrained and trained.

In [ ]:
# ── Weight matrices + stim activations per init_scale ─────────────────────
# Colormap limits are shared across trained/untrained within each figure
# so that weight magnitudes are directly comparable.

def _extract_weights(ac, H):
    n_ro  = ac.n_readout
    n_act = ac.policy_head.out_features
    W_in  = _t(ac.backbone.input2h.weight)
    W_rec = _t(ac.backbone.h2h.weight)
    W_val_r = _t(ac.value_head.weight)
    W_pol_r = _t(ac.policy_head.weight)
    W_val = np.full((H, 1), np.nan)
    W_val[:n_ro, 0] = W_val_r[0]
    W_pol = np.full((H, n_act), np.nan)
    W_pol[:n_ro, :] = W_pol_r.T
    return W_in, W_rec, W_val, W_pol, n_ro

for _isv in INIT_SCALE_VALS:
    _key = (_isv, SEED_PICK)
    if _key not in runs:
        print(f"Skipping init_scale={_isv} (no run)")
        continue
    _vd      = runs[_key]
    _H       = _vd["hidden_size"]
    _struct  = _vd["infer_trial_structure"]
    _stim_a  = np.array([t["stimulus"] for t in _struct])
    _n_stim_ = _vd["n_stimuli"]
    _stimuli_ = _vd["stimuli"]

    _run_dir   = SWEEP_DIR / _vd["run_id"]
    _init_path = _run_dir / "model_init.pt"
    _model_path = _run_dir / "model.pt"
    if not _model_path.exists() or not _init_path.exists():
        print(f"  Skipping init_scale={_isv}: model files missing")
        continue

    _ac_tr = load_model(_vd, torch.load(_model_path,  map_location="cpu"))
    _ac_un = load_model(_vd, torch.load(_init_path, map_location="cpu"))
    _hid_tr = np.array(_vd["infer_activations"]["hidden_states"])
    _hid_un = run_inference(_ac_un, _vd)

    _W_in_tr, _W_rec_tr, _W_val_tr, _W_pol_tr, _n_ro = _extract_weights(_ac_tr, _H)
    _W_in_un, _W_rec_un, _W_val_un, _W_pol_un, _     = _extract_weights(_ac_un, _H)
    _S_tr = stim_mean_act(_hid_tr, _stim_a, _struct, _n_stim_)
    _S_un = stim_mean_act(_hid_un, _stim_a, _struct, _n_stim_)

    _in_dim = _W_in_tr.shape[1]
    _n_act  = _W_pol_tr.shape[1]
    _col_w  = [_n_stim_, min(_in_dim, 10), min(_H, 20), 2, _n_act * 2]
    _ptitles = [
        f"Stim activation ({_n_stim_} stim)",
        f"W_in  (H×{_in_dim})",
        "W_rec  (H×H)",
        "W_val  (→ value)",
        f"W_pol  (→ {_n_act} actions)",
    ]

    # Shared colormap limits per panel type (combine trained + untrained data)
    _panel_lims = []
    for _pi, (_Dtr, _Dun, _div) in enumerate([
            (_S_tr, _S_un, False),
            (_W_in_tr, _W_in_un, True),
            (_W_rec_tr, _W_rec_un, True),
            (_W_val_tr, _W_val_un, True),
            (_W_pol_tr, _W_pol_un, True),
    ]):
        _combined = np.concatenate([_Dtr[np.isfinite(_Dtr)], _Dun[np.isfinite(_Dun)]])
        if _div:
            _lim = float(np.abs(_combined).max()) if len(_combined) else 1.0
            _panel_lims.append((-_lim, _lim))
        else:
            _panel_lims.append((0.0, float(np.nanpercentile(_combined, 99)) if len(_combined) else 1.0))

    _fig_w = sum(_col_w) * 18 / sum(_col_w) + 1.2
    fig = plt.figure(figsize=(_fig_w, 12))
    _outer = mgs.GridSpec(2, 1, figure=fig, hspace=0.35, top=0.90, bottom=0.05)

    _model_bundles = [
        ("Trained",   _W_in_tr, _W_rec_tr, _W_val_tr, _W_pol_tr, _S_tr),
        ("Untrained", _W_in_un, _W_rec_un, _W_val_un, _W_pol_un, _S_un),
    ]

    for _mi, (_mlbl, W_in, W_rec, W_val, W_pol, S_act) in enumerate(_model_bundles):
        _inner = mgs.GridSpecFromSubplotSpec(
            2, len(_ptitles) + 1,
            subplot_spec=_outer[_mi], hspace=0.08,
            height_ratios=[4, 1.2],
            width_ratios=_col_w + [0.25],
            wspace=0.18,
        )
        _panels_data = [
            S_act[_SORT_IDX, :],
            W_in[_SORT_IDX, :],
            W_rec[_SORT_IDX, :][:, _SORT_IDX],
            W_val[_SORT_IDX, :],
            W_pol[_SORT_IDX, :],
        ]
        _cmaps = ["hot", "RdBu_r", "RdBu_r", "RdBu_r", "RdBu_r"]

        for _pi, (_data, _cmap, (_vmin_, _vmax_), _ptitle) in enumerate(
                zip(_panels_data, _cmaps, _panel_lims, _ptitles)):
            _ax_h = fig.add_subplot(_inner[0, _pi])
            _ax_d = fig.add_subplot(_inner[1, _pi])

            _im = _ax_h.imshow(_data, aspect="auto", cmap=_cmap,
                               vmin=_vmin_, vmax=_vmax_, interpolation="nearest")

            if _pi in (3, 4) and _n_ro < _H:
                _ax_h.axhline(_n_ro - 0.5, color="cyan", lw=1.2, ls="--", alpha=0.8)
            if _pi == 2:
                for _by in _boundaries:
                    _ax_h.axhline(_by, color="white", lw=0.7, ls=":", alpha=0.6)
                    _ax_h.axvline(_by, color="white", lw=0.7, ls=":", alpha=0.6)

            _ax_h.set_title(_ptitle, fontsize=8, pad=2)
            _ax_h.tick_params(left=(_pi==0), labelleft=(_pi==0),
                              bottom=False, labelbottom=False)
            _ax_h.spines[["top","right","bottom","left"]].set_visible(False)
            if _pi == 0:
                _ax_h.set_yticks([0, _H//2, _H-1]); _ax_h.tick_params(labelsize=7)
                _ax_h.set_ylabel(f"{_mlbl}\n(n={_H} neurons)", fontsize=8)
                _ax_h.set_xticks(range(_n_stim_))
                _ax_h.set_xticklabels([f"s{i}" for i in range(_n_stim_)], fontsize=7)
                _ax_h.xaxis.set_tick_params(bottom=True, labelbottom=True)

            _valid = _data[np.isfinite(_data)]
            if len(_valid) > 0:
                _ax_d.hist(_valid, bins=40,
                           color="steelblue" if _cmap == "hot" else "slategray",
                           alpha=0.75, density=True, linewidth=0)
                if _cmap == "RdBu_r":
                    _ax_d.axvline(0, color="k", lw=0.8, ls="--", alpha=0.5)
                # shared x-range matches the heatmap colormap range
                _ax_d.set_xlim(_vmin_, _vmax_)
            _ax_d.tick_params(labelsize=7)
            _ax_d.spines[["top","right"]].set_visible(False)
            if _pi != 0:
                _ax_d.tick_params(labelleft=False)
            else:
                _ax_d.set_ylabel("density", fontsize=7)
            _ax_d.set_xlabel("act" if _pi == 0 else "weight", fontsize=7)

            if _pi == len(_ptitles) - 1:
                _cb_ax = fig.add_subplot(_inner[0, -1])
                fig.colorbar(_im, cax=_cb_ax)
                _cb_ax.tick_params(labelsize=6)
                fig.add_subplot(_inner[1, -1]).set_visible(False)

    fig.suptitle(
        f"Weights & stim activations — init_scale={_isv}  [seed={SEED_PICK}]\n"
        f"Shared colormap ranges across trained/untrained  |  "
        f"cyan = readout boundary (n_ro={_n_ro})",
        fontsize=10,
    )
    fig.savefig(out_dir / f"fig4_weights_is{str(_isv).replace('.','p')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()


## §6  Weight norm evolution per selectivity subgroup

For the representative seed (`SEED_PICK`) of each `init_scale`, defines neuron subgroups
from trained-model selectivity (argmax stim response, SI-range threshold), then tracks
mean per-neuron weight norms across checkpoints.
Panels: W_in, W_rec outgoing, W_rec incoming, W_readout (val+pol).

In [ ]:
# ── Weight norm evolution per selectivity subgroup — init_scale sweep ────────
# One row per init_scale (SEED_PICK run), 4 panels: W_in, W_rec_out, W_rec_in,
# W_readout. Subgroups defined from the trained model's stim selectivity.

_SI_THR  = 0.1
_SIL_THR = 1e-4

def _compute_subgroups_from_vd(vd):
    """Return (group_idx_list, grp_labels, group_sizes) from vis_data."""
    act    = vd["infer_activations"]
    stim_a = act["stimulus"]
    hs     = act["hidden_states"]
    struct = act["trial_structure"]
    ns     = vd["n_stimuli"]
    means  = []
    for si in range(ns):
        mask   = stim_a == si
        chunks = np.concatenate([
            hs[t["stim_window"][0]:t["stim_window"][1]]
            for t, m in zip(struct, mask) if m
        ], axis=0)
        means.append(chunks.mean(axis=0))
    means   = np.stack(means, axis=1)   # (H, ns)
    max_abs = np.abs(means).max(axis=1)
    pref    = means.argmax(axis=1)
    si_rng  = (means.max(1) - means.min(1)) / (np.abs(means).sum(1) + 1e-8)
    sel     = (si_rng >= _SI_THR) & (max_abs >= _SIL_THR)
    g_idx, g_sz = [], []
    for si in range(ns):
        grp = np.where(sel & (pref == si))[0]
        grp = grp[np.argsort(-si_rng[grp])]
        g_idx.append(grp); g_sz.append(len(grp))
    nonsel = np.where(~sel)[0]
    g_idx.append(nonsel); g_sz.append(len(nonsel))
    lbls = list(vd.get("stimuli", [f"s{i}" for i in range(ns)])) + ["non-sel"]
    return g_idx, lbls, g_sz

def _grp_norms_from_sd(sd, vd, g_idx, lbls):
    obs_d = vd["infer_states"].shape[1] + 2
    H     = vd["hidden_size"]
    rf    = vd.get("readout_fraction", 1.0)
    bb = RNN(input_size=obs_d, hidden_size=H, output_size=1)
    ac = ActorCritic(backbone=bb, num_actions=2, readout_fraction=rf)
    ac.load_state_dict(sd)
    ro   = ac.n_readout
    Win  = np.array(ac.backbone.input2h.weight.detach().cpu().tolist(), dtype=np.float32)
    Wrec = np.array(ac.backbone.h2h.weight.detach().cpu().tolist(),     dtype=np.float32)
    Wval = np.array(ac.value_head.weight.detach().cpu().tolist(),       dtype=np.float32)
    Wpol = np.array(ac.policy_head.weight.detach().cpu().tolist(),      dtype=np.float32)
    out = {}
    for idx, lbl in zip(g_idx, lbls):
        if not len(idx):
            out[lbl] = [np.nan]*4; continue
        ridx = idx[idx < ro]
        out[lbl] = [
            np.linalg.norm(Win[idx,  :], axis=1).mean(),
            np.linalg.norm(Wrec[:, idx], axis=0).mean(),
            np.linalg.norm(Wrec[idx, :], axis=1).mean(),
            ((np.linalg.norm(Wval[:, ridx]) + np.linalg.norm(Wpol[:, ridx]))
             / max(len(ridx), 1) if len(ridx) else np.nan),
        ]
    return out

_wn_panels = [
    "W_in (input→neuron)",
    "W_rec outgoing (neuron→others)",
    "W_rec incoming (others→neuron)",
    "W_readout (neuron→heads)",
]

_n_scales = len(INIT_SCALE_VALS)
fig, axes = plt.subplots(_n_scales, 4, figsize=(16, 3.2 * _n_scales), sharey=False)
if _n_scales == 1:
    axes = axes.reshape(1, 4)

for _ri, _isv in enumerate(INIT_SCALE_VALS):
    _vd = runs.get((_isv, SEED_PICK))
    if _vd is None:
        for ax in axes[_ri]: ax.text(0.5, 0.5, "no data", ha="center", va="center",
                                     transform=ax.transAxes)
        continue

    _run_dir_wn = SWEEP_DIR / _vd["run_id"]
    _ckpts_wn   = sorted((_run_dir_wn / "checkpoints").glob("checkpoint_*.pt"))
    if not _ckpts_wn:
        for ax in axes[_ri]: ax.text(0.5, 0.5, "no ckpts", ha="center", va="center",
                                     transform=ax.transAxes)
        continue

    _g_idx, _lbls, _g_sz = _compute_subgroups_from_vd(_vd)
    _ckpt_tn = [int(p.stem.split("_")[1]) for p in _ckpts_wn]
    _wn_h    = {lbl: [[] for _ in range(4)] for lbl in _lbls}

    print(f"  is={_isv}: {len(_ckpts_wn)} ckpts, "
          f"groups={[f'{l}:{len(i)}' for l,i in zip(_lbls,_g_idx)]}")
    for _cf in _ckpts_wn:
        _ns = _grp_norms_from_sd(
            torch.load(_cf, map_location="cpu", weights_only=True), _vd, _g_idx, _lbls
        )
        for lbl in _lbls:
            for pi in range(4):
                _wn_h[lbl][pi].append(_ns[lbl][pi])

    # Final trained model weights
    _final_sd = torch.load(_run_dir_wn / "model.pt", map_location="cpu", weights_only=True)
    _ns_fin   = _grp_norms_from_sd(_final_sd, _vd, _g_idx, _lbls)

    _cols = STIM_COLORS[:len(_lbls) - 1] + ["grey"]
    for pi, ax in enumerate(axes[_ri]):
        for lbl, col in zip(_lbls, _cols):
            ax.plot(_ckpt_tn, _wn_h[lbl][pi], color=col, lw=1.8,
                    marker="o", ms=3, label=lbl)
            ax.scatter([_ckpt_tn[-1]], [_ns_fin[lbl][pi]],
                       color=col, s=50, zorder=5, marker="*")
        ax.set_xlabel("Training trial", fontsize=8)
        ax.set_title(_wn_panels[pi], fontsize=8) if _ri == 0 else None
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if pi == 0:
            ax.set_ylabel(f"is={_isv}\nMean norm", fontsize=8)
        if pi == 3 and _ri == 0:
            ax.legend(fontsize=7, title="Subgroup", title_fontsize=6)

fig.suptitle(
    f"Weight norm per selectivity subgroup over training  [seed={SEED_PICK}]\n"
    f"Stars = final trained model",
    fontsize=11, y=1.01,
)
plt.tight_layout()
fig.savefig(out_dir / "fig6_weight_norm_evolution.png", dpi=150, bbox_inches="tight")
plt.show()

## §5 Cross-scale summary

Bar/line plots of performance, selectivity and activation magnitude across all seeds.

In [ ]:
# ── Cross-scale summary: performance + selectivity + activation magnitude ──
_metrics = [
    ("spearman_r",       "Spearman r\n(lick–value)",    (-1, 1)),
    ("psa_score",        "PSA score",                    (0, 1)),
    ("lick_high",        "Lick rate — high stim",        (0, 1)),
    ("lick_mid",         "Lick rate — mid stim",         (0, 1)),
    ("lick_low",         "Lick rate — low stim",         (0, 1)),
    ("mean_init_act",    "Mean init activation",         (None, None)),
    ("mean_trained_act", "Mean trained activation",      (None, None)),
]

fig, axes = plt.subplots(1, len(_metrics), figsize=(3.5 * len(_metrics), 4))
_xs = [str(v) for v in INIT_SCALE_VALS]

for ax, (col, title, (ylo, yhi)) in zip(axes, _metrics):
    _means, _sems = [], []
    for _isv in INIT_SCALE_VALS:
        _vals = df.loc[df.init_scale == _isv, col].dropna()
        _means.append(_vals.mean() if len(_vals) else np.nan)
        _sems.append(_vals.sem()   if len(_vals) else np.nan)
    ax.bar(_xs, _means, yerr=_sems, capsize=4, color="steelblue", alpha=0.8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("init_scale", fontsize=8)
    ax.tick_params(labelsize=8)
    if ylo is not None:
        ax.set_ylim(ylo, yhi)

n_seeds = df["seed"].nunique()
fig.suptitle(
    f"Cross-scale summary  (mean ± SEM, n≤{n_seeds} seeds per scale)",
    fontsize=11, y=1.02,
)
plt.tight_layout()
fig.savefig(out_dir / "fig5_cross_scale_summary.png", dpi=150, bbox_inches="tight")
plt.show()

# Also: init vs trained activation side-by-side
fig2, ax2 = plt.subplots(figsize=(7, 4))
_x = np.arange(len(INIT_SCALE_VALS))
_w = 0.35
_im_vals  = [df.loc[df.init_scale==v,"mean_init_act"].mean()    for v in INIT_SCALE_VALS]
_im_sems  = [df.loc[df.init_scale==v,"mean_init_act"].sem()     for v in INIT_SCALE_VALS]
_tr_vals  = [df.loc[df.init_scale==v,"mean_trained_act"].mean() for v in INIT_SCALE_VALS]
_tr_sems  = [df.loc[df.init_scale==v,"mean_trained_act"].sem()  for v in INIT_SCALE_VALS]
ax2.bar(_x - _w/2, _im_vals, _w, yerr=_im_sems,  capsize=4, label="init",    alpha=0.8)
ax2.bar(_x + _w/2, _tr_vals, _w, yerr=_tr_sems, capsize=4, label="trained", alpha=0.8)
ax2.set_xticks(_x); ax2.set_xticklabels(_xs)
ax2.set_xlabel("init_scale"); ax2.set_ylabel("mean hidden activation")
ax2.set_title("Init vs trained activation magnitude")
ax2.legend()
plt.tight_layout()
fig2.savefig(out_dir / "fig5b_activation_magnitude.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Final lick probabilities per stimulus across init_scale ───────────────
# Two panels:
#   Left:  lick prob vs init_scale — one line per stimulus (mean ± SEM across seeds)
#   Right: per-seed scatter with means, one column of dots per (scale × stim)

_stim_labels  = ["s0 (0%)", "s1 (50%)", "s2 (100%)"]
_lick_cols    = ["lick_low", "lick_mid", "lick_high"]   # matches s0, s1, s2
_ideal_licks  = [0.0, 0.5, 1.0]                         # perfect value-matching

fig, (ax_line, ax_dot) = plt.subplots(1, 2, figsize=(11, 4))

_xs      = np.arange(len(INIT_SCALE_VALS))
_xs_str  = [str(v) for v in INIT_SCALE_VALS]
_offsets = np.linspace(-0.25, 0.25, len(_stim_labels))

for _ci, (_col, _lbl, _ideal) in enumerate(zip(_lick_cols, _stim_labels, _ideal_licks)):
    _means, _sems, _all_vals = [], [], []
    for _isv in INIT_SCALE_VALS:
        _vals = df.loc[df.init_scale == _isv, _col].dropna().values
        _means.append(_vals.mean() if len(_vals) else np.nan)
        _sems.append(_vals.std() / np.sqrt(len(_vals)) if len(_vals) > 1 else 0.0)
        _all_vals.append(_vals)
    _means = np.array(_means)
    _sems  = np.array(_sems)
    _col_c = STIM_COLORS[_ci % len(STIM_COLORS)]

    # ── left: line + SEM band ────────────────────────────────────────
    ax_line.plot(_xs, _means, color=_col_c, lw=2.0, marker="o", ms=5, label=_lbl)
    ax_line.fill_between(_xs, _means - _sems, _means + _sems,
                         color=_col_c, alpha=0.20, linewidth=0)
    # Ideal dashed reference
    ax_line.axhline(_ideal, color=_col_c, lw=0.8, ls=":", alpha=0.5)

    # ── right: per-seed dots + mean bar ──────────────────────────────
    for _xi, (_isv, _vals) in enumerate(zip(INIT_SCALE_VALS, _all_vals)):
        _jx = _xs[_xi] + _offsets[_ci]
        ax_dot.scatter(np.full(len(_vals), _jx), _vals,
                       color=_col_c, s=18, alpha=0.65, zorder=3)
        if len(_vals):
            ax_dot.plot([_jx - 0.06, _jx + 0.06],
                        [_vals.mean(), _vals.mean()],
                        color=_col_c, lw=2.0, zorder=4)

for ax in (ax_line, ax_dot):
    ax.set_xticks(_xs)
    ax.set_xticklabels(_xs_str, fontsize=9)
    ax.set_xlabel("init_scale", fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Lick probability", fontsize=9)
    ax.axhline(0, color="k", lw=0.5, alpha=0.3)
    ax.axhline(1, color="k", lw=0.5, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(labelsize=8)

ax_line.set_title("Mean ± SEM across seeds\n(dotted = ideal value-matched policy)", fontsize=9)
ax_dot.set_title("Per-seed lick probabilities\n(horizontal bar = seed mean)", fontsize=9)
ax_line.legend(fontsize=8, loc="center right")

# Add legend for stim colours on dot plot
for _ci, _lbl in enumerate(_stim_labels):
    ax_dot.plot([], [], color=STIM_COLORS[_ci], lw=2, label=_lbl)
ax_dot.legend(fontsize=8, loc="center right")

fig.suptitle(
    f"Final lick probabilities — trained models  (n≤{df['seed'].nunique()} seeds per scale)",
    fontsize=11, y=1.02,
)
plt.tight_layout()
fig.savefig(out_dir / "fig6_lick_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
